# 04 · Capa gold
Una fila por cliente: `application` limpia + agregados de las tablas secundarias + ratios + partición 60/20/20 estratificada por `TARGET`. Ninguna variable de esta capa usa `TARGET` para calcularse.

In [0]:
# Se usa left join para conservar a los 307,511 clientes aunque no tengan historial:
from pyspark.sql import functions as F, Window

ESQ = "workspace.credit_risk"

gold = spark.table(f"{ESQ}.silver_application")
for t in ["silver_inst_agg", "silver_bureau_agg", "silver_prev_agg",
          "silver_cc_agg", "silver_pos_agg", "silver_bb_agg"]:
    gold = gold.join(spark.table(f"{ESQ}.{t}"), "SK_ID_CURR", "left")

# En los conteos, "sin historial" significa 0; el resto queda como nulo
conteos = ["INST_N_CUOTAS", "BUR_N_CREDITOS", "BUR_N_ACTIVOS", "BUR_N_CERRADOS",
           "PREV_N_SOLICITUDES", "PREV_N_APROBADAS", "PREV_N_RECHAZADAS",
           "CC_N_TARJETAS", "POS_N_REGISTROS", "BB_MESES_TOTAL"]
gold = gold.fillna(0, subset=conteos)

In [0]:
# ratios de capacidad de pago. Son variables clásicas de riesgo (qué parte del ingreso se va en la cuota, cuánto se financia respecto al bien):
def razon(num, den):
    return F.when(F.col(den) > 0, F.col(num) / F.col(den))

gold = (gold
    .withColumn("CREDIT_INCOME_RATIO",  razon("AMT_CREDIT", "AMT_INCOME_TOTAL"))
    .withColumn("ANNUITY_INCOME_RATIO", razon("AMT_ANNUITY", "AMT_INCOME_TOTAL"))
    .withColumn("ANNUITY_CREDIT_RATIO", razon("AMT_ANNUITY", "AMT_CREDIT"))
    .withColumn("CREDIT_GOODS_RATIO",   razon("AMT_CREDIT", "AMT_GOODS_PRICE"))
    .withColumn("INCOME_PER_PERSON",    razon("AMT_INCOME_TOTAL", "CNT_FAM_MEMBERS"))
    .withColumn("EMPLOYED_AGE_RATIO",   razon("EMPLOYED_YEARS", "AGE_YEARS")))

# Promedio de los puntajes externos disponibles (ignora los nulos)
ext = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
n_ext = sum(F.col(c).isNotNull().cast("int") for c in ext)
s_ext = sum(F.coalesce(F.col(c), F.lit(0.0)) for c in ext)
gold = gold.withColumn("EXT_SOURCE_MEAN", F.when(n_ext > 0, s_ext / n_ext))

In [0]:
# Partición y guardado:
w = Window.partitionBy("TARGET").orderBy(F.xxhash64("SK_ID_CURR"), "SK_ID_CURR")

gold = (gold
    .withColumn("_pr", F.percent_rank().over(w))
    .withColumn("split", F.when(F.col("_pr") < 0.6, "train")
                          .when(F.col("_pr") < 0.8, "valid")
                          .otherwise("test"))
    .drop("_pr"))

gold.write.mode("overwrite").saveAsTable(f"{ESQ}.gold_features")

In [0]:
# Punto de control:
g = spark.table(f"{ESQ}.gold_features")
n, d = g.count(), g.select("SK_ID_CURR").distinct().count()
print(f"gold_features: {n:,} filas, {d:,} clientes, {len(g.columns)} columnas -> {'OK' if n == d == 307511 else 'REVISAR'}")

display(g.groupBy("split")
         .agg(F.count("*").alias("n"), F.sum("TARGET").alias("defaults"), F.avg("TARGET").alias("tasa_default"))
         .orderBy("split"))